In [ ]:
import random as rd
import pygraphviz as pgv
import json
map_data = pgv.AGraph("map_data.dot")
entanglements = pgv.AGraph("entanglements.dot")
with open('units.json') as ujson:
    units = json.load(ujson)
with open('orders.json') as ojson:
    orders = json.load(ojson)

def compat(utype,tgtype):
    return tgtype == 'coast' or (tgtype == 'land' and utype == 'Army') or (tgtype == 'sea' and utype == 'Fleet')

def adj(origin,destination):
    return map_data.has_neighbor(origin,destination)

def supincl(unit,origin):
    return origin in unit['superposition']

#Handling order contradictions
filter_dict = {}
invalid_keys = []
for order in orders:
    if order['unit'] not in filter_dict:
        filter_dict[order['unit']] = order['type']
    elif filter_dict[order['unit']] != order['type']:
        invalid_keys.append(order['unit'])
invalid_keys = list(set(invalid_keys))
if invalid_keys:
    print("Incompatible orders issued to the following units : " + str(invalid_keys))
orders = [val for val in orders if val['unit'] not in invalid_keys]
for i in invalid_keys:
    orders.append({"type" : "H", "unit" : i, "origin" : "", "destination" : "", "score" : 0, "convoyed" : False, "pointer" : None})

#Handling invalid province specifications. MAKE SURE CONVOYS ARE SET AT THE BEGINNING OF THE JSON FILE. Holding orders must be duplicated for every province support held.
for order in orders:
    match order['type']:
        case 'S':
            if order['pointer'] == None:
                print("No matching order to support.")
                order['type'] = 'H'
                order['origin'] = ''
                order['destination'] = ''
            elif not supincl(units[order['unit']],order['origin']):
                print(str(order['unit']) + " has no mass in " + str(orders['origin']))
                order['type'] = 'H'
                order['origin'] = ''
                order['destination'] = ''
            elif not adj(order['origin'],order['destination']) or not compat(units[order['unit']]['type'],map_data.get_node(order['destination']).attr['type']):
                print(str(order['unit']) + " cannot support a unit to " + str(order['destination']))
                order['type'] = 'H'
                order['origin'] = ''
                order['destination'] = ''
            elif orders[order['pointer']]['type'] == 'H' and not supincl(units[orders[order['pointer']]['unit']],order['destination']):
                print("Cannot support " + str(orders[order['pointer']]['unit']) + " with no mass in " + str(order['destination']))
                order['type'] = 'H'
                order['origin'] = ''
                order['destination'] = ''
            elif order['destination'] != orders[order['pointer']]['destination']:
                print("Invalid support (order mismatch)")
                order['type'] = 'H'
                order['origin'] = ''
                order['destination'] = ''
        case 'T':
            if not supincl(units[order['unit']],order['origin']):
                print(str(order['unit']) + " has no mass in " + str(orders['origin']))
                order['type'] = 'H'
                order['origin'] = ''
                order['destination'] = ''
            elif not compat(units[order['unit']]['type'],map_data.get_node(order['destination']).attr['type']):
                print(str(order['unit']) + " cannot tunnel to " + str(order['destination']))
                order['type'] = 'H'
                order['origin'] = ''
                order['destination'] = ''
            elif not adj(order['origin'],order['destination']) and not order['convoyed']:
                print(str(order['unit']) + " cannot tunnel to " + str(order['destination']))
                order['type'] = 'H'
                order['origin'] = ''
                order['destination'] = ''
        case 'C':
            if order['pointer'] == None:
                print("No matching order to convoy.")
                order['type'] = 'H'
            elif not orders[order['pointer']]['convoyed']:
                print("Invalid convoy (order mismatch)")
                order['type'] = 'H'
            elif not supincl(units[order['unit']],orders[order['pointer']]['origin']):
                print("Invalid convoy : " + str(order['unit']) + " and " + str(orders[order['pointer']]['unit']) + " do not overlap")
                order['type'] = 'H'
                orders[order['pointer']]['convoyed'] = False
        case 'M':
            if units[order['unit']]['mflag']:
                print("Cannot measure " + str(order['unit']) + " two turns in a row")
                order['type'] = 'H'
                order['destination'] = ''
                units[order['unit']]['mflag'] = False
            elif not supincl(units[order['unit']],order['destination']):
                print(str(order['unit']) + " has no mass in " + str(order['destination']))
                order['type'] = 'H'
                order['destination'] = ''
        case 'H':
            print('')
        case _:
            print("Order type not specified. Defaulting to hold.")
            order['type'] = 'H'
            order['origin'] = ''
            order['destination'] = ''

#Handling cut supports and attributing support values to their corresponding order + support-induced entanglements
atkd_provinces = []
for order in orders:
    if order['type'] == 'T':
        atkd_provinces.append(order['destination'])
for order in orders:
    if order['type'] == 'S' and (order['origin'] not in atkd_provinces):
        sup_value = units[order['unit']]['superposition'].count(order['origin']) * (100 // len(units[order['unit']]['superposition']))
        orders[order['pointer']]['score'] += sup_value
        entanglements.add_edge(orders[order['pointer']]['unit'],order['unit'])
    elif order['type'] == 'S' and (order['origin'] in atkd_provinces):
        print(str(order['unit']) + "'s support from " + str(order['origin']) + " was cut")

#Handling holds
for order in orders:
    if order['type'] == 'H':
        for attack in orders:
            if (attack['type'] == 'T' or attack['type'] == 'M') and supincl(units[order['unit']],attack['destination']):
                attack['score'] -= order['score'] + units[order['unit']]['superposition'].count(attack['destination']) * (100 // len(units[order['unit']]['superposition']))

#Handling tunnel and measuring success
success_keys = []
for order in orders:
    if order['type'] == 'T' or order['type'] == 'M':
        order['score'] += units[order['unit']]['superposition'].count(order['origin']) * (100 // len(units[order['unit']]['superposition']))
        roll = rd.randint(1,100)
        result = order['score'] - roll
        if result < 0:
            print(str(order['unit']) + " attempt failed with success rate = " + str(order['score']) + " from roll = " + str(roll))
            if order['type'] == 'M' : units[order['unit']]['superposition'] = list(filter((order['destination']).__ne__, units[order['unit']]['superposition']))
        else:
            print(str(order['unit']) + " attempt succeeded with success rate = " + str(order['score']) + " from roll = " + str(roll))
            success_keys.append(order)
            if order['type'] == 'M' and order['unit'] in entanglements.nodes():
                for i in entanglements.neighbours(order['unit']):
                    print(str(i) + " needs to be measured as a result of entanglement with " + str(orders['unit']))

#Handling bounces
provinces = map_data.nodes()
provinces = {value: 0 for value in provinces}
for unit in units.keys():
    for n in units[unit]['superposition']:
        provinces[n] += units[unit]['superposition'].count(n) * (100 // len(units[unit]['superposition']))
invasion = []
for order_i in success_keys:
    for order_j in success_keys.copy():
        if order_i != order_j:
            weight_i = 100 // (1 + len(units[order_i['unit']]['superposition']))
            weight_j = 100 // (1 + len(units[order_j['unit']]['superposition']))
            if (order_i['destination'] == order_j['destination']) and ((weight_i + weight_j + provinces[order_i['destination']]) > 100) :
                print(str(order_i['unit']) + " and " + str(order_j['unit']) + " bounced in " + str(order_i['destination']))
            else:
                if order_i['type'] == 'M': 
                    units[order_i['unit']]['superposition'] = [order_i['destination']]
                    units[order_i['unit']]['mflag'] = True
                    invasion.append((order_i['destination'],order_i['unit']))
                elif order_i['type'] == 'T' :
                    units[order_i['unit']]['superposition'].append(order_i['destination'])
                    invasion.append((order_i['destination'],order_i['unit']))

#Handling retreats
provinces = map_data.nodes()
provinces = {value: 0 for value in provinces}
for unit in units.keys():
    for n in units[unit]['superposition']:
        provinces[n] += units[unit]['superposition'].count(n) * (100 // len(units[unit]['superposition']))
reserve = 0
for province, immune in invasion:
    for unit in units.keys():
        if unit != immune and province in units[unit]['superposition'] and provinces[province] > 100:
            print(str(unit) + " was dislodged from " + str(province))
            units[unit]['superposition'] = list(filter((province).__ne__, units[unit]['superposition']))
        if not units[unit]['superposition']:
            del units[unit]
        else :
            reserve += 100 % len(units[unit]['superposition'])
print("Reserve at " + str(reserve) + "% capacity")

with open("units.json", "w") as f:
    json.dump(units, f)

2D attempt succeeded with success rate = 100 from roll = 18
2A attempt succeeded with success rate = 100 from roll = 85
Reserve at 0% capacity
